In [23]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.Scaffolds import MurckoScaffold
from itertools import accumulate, chain

In [8]:
data = pd.read_csv('Path to data CSV file')

In [22]:
def standardize_mol(mol):
    clean_mol = rdMolStandardize.Cleanup(mol)
    parent_mol = rdMolStandardize.FragmentParent(clean_mol)
    uncharger = rdMolStandardize.Uncharger()
    uncharged_mol = uncharger.uncharge(parent_mol)
    return uncharged_mol

def get_canonical_smiles(mol):
    return Chem.MolToSmiles(mol, isomericSmiles=True, canonical=True)

def find_duplicates(smiles_df, smiles_col='smiles', id_col='cid'):
    canonical_smiles_dict = {}
    duplicates = []
    unique_smiles_with_ids = []

    for index, row in smiles_df.iterrows():
        id = row[id_col]
        smiles = row[smiles_col]
        mol = Chem.MolFromSmiles(smiles)
        
        if mol is not None:
            std_mol = standardize_mol(mol)
            canonical_smiles = get_canonical_smiles(std_mol)
            
            if canonical_smiles in canonical_smiles_dict:
                # Append the ID of the duplicate
                duplicates.append((id, canonical_smiles_dict[canonical_smiles]))
            else:
                # Store the unique SMILES along with its ID
                unique_smiles_with_ids.append((id, canonical_smiles))
                canonical_smiles_dict[canonical_smiles] = id

    return unique_smiles_with_ids, duplicates

In [14]:
unique_smiles, duplicates = find_duplicates(data, smiles_col='smiles', id_col='cid')

In [ ]:
print(f'The number of unique smiles is {len(unique_smiles)}')
print(f'The number of duplicates smiles is {len(duplicates)}')

In [24]:
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator
from rdkit.ML.Cluster import Butina
import numpy as np
import pandas as pd
from typing import Tuple

def similarity_constrained_split(
    df: pd.DataFrame,
    smiles_col: str = "smiles",
    test_size: float = 0.2,
    threshold: float = 0.50,        # max allowed train–test similarity
    fp_radius: int = 2,             # ECFP4 => radius=2
    fp_nbits: int = 2048,
    random_state: int = 42,
    drop_invalid: bool = True,
) -> Tuple[pd.DataFrame, pd.DataFrame, dict]:
    """
    Split df into (train, test) so that no cross-set Tanimoto pair >= threshold
    (approximately guaranteed by Butina clustering on 1 - similarity).

    Returns:
        train_df, test_df, info (dict with verification stats)
    """
    rng = np.random.default_rng(random_state)

    # --- 1) SMILES -> Mol (handle invalids) ---
    mols = []
    valid_idx = []
    invalid_idx = []
    for i, smi in enumerate(df[smiles_col].astype(str).values):
        m = Chem.MolFromSmiles(smi)
        if m is None:
            invalid_idx.append(i)
            if not drop_invalid:
                raise ValueError(f"Invalid SMILES at index {i}: {smi}")
        else:
            mols.append(m)
            valid_idx.append(i)

    if invalid_idx:
        print(f"[info] Dropped {len(invalid_idx)} invalid SMILES." if drop_invalid else
              f"[warn] Found {len(invalid_idx)} invalid SMILES.")

    df_valid = df.iloc[valid_idx].reset_index(drop=True)
    n = len(mols)
    if n == 0:
        raise ValueError("No valid molecules to split.")

    # --- 2) Fingerprints (Morgan / ECFP4) ---
    morgan = rdFingerprintGenerator.GetMorganGenerator(radius=fp_radius, fpSize=fp_nbits)
    fps = [morgan.GetFingerprint(m) for m in mols]

    # --- 3) Pairwise distances for Butina (upper triangle) ---
    # distance = 1 - Tanimoto; we cluster with cutoff = 1 - threshold
    dists = []
    for i in range(1, n):
        sims = DataStructs.BulkTanimotoSimilarity(fps[i], fps[:i])  # similarities to 0..i-1
        dists.extend([1.0 - s for s in sims])

    cutoff = 1.0 - threshold
    # old: clusters = Butina.ClusterData(dists, nPts=n, cutoff=cutoff, isDistData=True)
    clusters = Butina.ClusterData(dists, n, cutoff, isDistData=True)

    # Shuffle cluster order for randomness/reproducibility
    clusters = list(clusters)
    rng.shuffle(clusters)

    # --- 4) Assign whole clusters to train/test to match size ratio ---
    target_test = int(round(test_size * n))
    test_idx, train_idx = [], []
    cur_test = 0
    for cl in clusters:
        # Greedy: put cluster where it moves us closer to target_test
        if abs((cur_test + len(cl)) - target_test) <= abs(cur_test - target_test):
            test_idx.extend(cl)
            cur_test += len(cl)
        else:
            train_idx.extend(cl)

    # Fallback: if one side empty (rare), move the largest cluster
    if len(test_idx) == 0:
        largest = max(clusters, key=len)
        test_idx = list(largest)
        train_idx = [i for i in range(n) if i not in test_idx]
    if len(train_idx) == 0:
        largest = max(clusters, key=len)
        train_idx = list(largest)
        test_idx = [i for i in range(n) if i not in train_idx]

    train_df = df_valid.iloc[train_idx].reset_index(drop=True)
    test_df  = df_valid.iloc[test_idx].reset_index(drop=True)

    # --- 5) Verify cross-set similarity stats ---
    # Compute test x train similarity matrix in blocks to save memory if needed
    n_test, n_train = len(test_idx), len(train_idx)
    total_pairs = n_test * n_train
    ge_count = 0

    # Count pairs >= threshold (no need to store full matrix)
    for i in range(n_test):
        sims = DataStructs.BulkTanimotoSimilarity(fps[test_idx[i]], [fps[j] for j in train_idx])
        ge_count += np.count_nonzero(np.array(sims) >= threshold)

    pct_pairs_ge = 100.0 * ge_count / total_pairs if total_pairs > 0 else 0.0

    # Coverage per side
    test_with_match = 0
    train_with_flags = np.zeros(n_train, dtype=bool)

    for i in range(n_test):
        sims = DataStructs.BulkTanimotoSimilarity(fps[test_idx[i]], [fps[j] for j in train_idx])
        mask = np.array(sims) >= threshold
        if mask.any():
            test_with_match += 1
            train_with_flags |= mask

    train_with_match = int(train_with_flags.sum())
    pct_test_with = 100.0 * test_with_match / n_test if n_test > 0 else 0.0
    pct_train_with = 100.0 * train_with_match / n_train if n_train > 0 else 0.0

    info = {
        "n_valid": n,
        "n_train": n_train,
        "n_test": n_test,
        "threshold": threshold,
        "total_pairs": total_pairs,
        "pairs_ge_threshold": ge_count,
        "pct_pairs_ge_threshold": pct_pairs_ge,
        "test_with_match_ge_threshold": f"{test_with_match}/{n_test}",
        "train_with_match_ge_threshold": f"{train_with_match}/{n_train}",
        "pct_test_with_match_ge_threshold": pct_test_with,
        "pct_train_with_match_ge_threshold": pct_train_with,
        "n_clusters": len(clusters),
        "avg_cluster_size": float(np.mean([len(c) for c in clusters])) if clusters else 0.0,
    }

    return train_df, test_df, info


In [ ]:
train_df, test_df, stats = similarity_constrained_split(
    df_train_test,               # your DataFrame
    smiles_col="smiles",
    test_size=0.20,
    threshold=0.50,     # <- enforce < 0.50 cross-set similarity
    random_state=42
)

print(stats)

In [ ]:
# Save the DataFrames to CSV files
data_train.to_csv('Path to csv data_train.csv', index=False)
data_test.to_csv('Path to csv data_test.csv', index=False)

In [ ]:
mol_train = [Chem.MolFromSmiles(x) for x in data_train['smiles']]
mol_test= [Chem.MolFromSmiles(x) for x in data_test['smiles']]


fp_train= [AllChem.GetMorganFingerprintAsBitVect(x,radius=2,nBits=2048) for x in mol_train]

fp_test= [AllChem.GetMorganFingerprintAsBitVect(x,radius=2,nBits=2048) for x in mol_test]


size_x= len(fp_train)
size_y= len(fp_test)

print(size_x)
print(size_y)


similarity_matrix = np.zeros((size_y, size_x))
similarity_matrix.shape

idx = 0
np_fps = list()
for fp in fp_test:
    np_fp = np.zeros((1,))
    Chem.DataStructs.ConvertToNumpyArray(fp, np_fp)
    np_fps.append(np_fp)
    # Calculate Tanimoto similarity
    similarity = Chem.DataStructs.BulkTanimotoSimilarity(fp, fp_train)
    # Save it to similarity matrix
    similarity_matrix[idx] = similarity
    idx += 1


df_similarity = pd.DataFrame(similarity_matrix)

df_similarity = pd.DataFrame(similarity_matrix)
df_similarity.columns = list(data_train['mol_id'])
df_similarity.index = data_test['mol_id']
df_similarity


import seaborn as sns
import matplotlib.pyplot as plt
sns.set(rc={'figure.figsize':(11.7,8.27)})
fig, ax = plt.subplots(dpi=300, figsize=(7,5))
ax = sns.heatmap(df_similarity, vmin=0, vmax=1,
                yticklabels=False, xticklabels=False,cmap="coolwarm")
ax.set_xlabel("Train", fontsize = 15)
ax.set_ylabel("Test", fontsize = 15)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Extract upper triangle values excluding the diagonal
tanimoto_coefficients = np.array(similarity_matrix)[np.triu_indices_from(np.array(similarity_matrix), k=1)]

# Define the ranges for Tanimoto coefficients
ranges = np.arange(0, 1.1, 0.1)
accumulated_proportion = []

# Calculate accumulated proportion for each range
for i in ranges:
    proportion = np.sum(tanimoto_coefficients < i) / len(tanimoto_coefficients)
    accumulated_proportion.append(proportion)

# Create a single figure
fig, ax_tanimoto = plt.subplots(figsize=(8, 6))

# Plotting the data
ax_tanimoto.plot(ranges, accumulated_proportion, 'o-', color='blue', linewidth=2, markersize=8, label='Accumulated Proportion')

# Set axis labels with larger font size
ax_tanimoto.set_xlabel('Tanimoto Coefficients Range', fontsize=16, fontweight='bold')
ax_tanimoto.set_ylabel('Accumulated Proportion', fontsize=16, fontweight='bold')

# Set the title with larger font size
ax_tanimoto.set_title('Tanimoto Coefficients Between Training and Test Active Scaffold', 
          fontsize=18, fontweight='bold')

# Customize x-ticks (improved formatting)
x_labels = [f'[{i:.1f}, {i + 0.1:.1f})' for i in np.arange(0, 1, 0.1)] + ['[1.0, 1.0]']
ax_tanimoto.set_xticks(ranges)
ax_tanimoto.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=14, color='black')

# Customize y-ticks
ax_tanimoto.tick_params(axis='y', labelsize=14, colors='black')

# Add grid lines for better readability
ax_tanimoto.grid(True, linestyle='--', alpha=0.7)

# Add black borders to all four sides
for spine in ax_tanimoto.spines.values():
    spine.set_edgecolor('black')
    spine.set_linewidth(2)

# Customize major ticks (improved consistency)
ax_tanimoto.tick_params(axis='both', which='major', direction='in', length=8, width=2, colors='black')

# Add a legend
ax_tanimoto.legend(fontsize=14, loc='lower right')

# Show the plot
plt.tight_layout()
plt.show()
